In [23]:
# ---------------- Imports ----------------
import os
import json
from collections import defaultdict
import yaml
import pandas as pd
from transformers import AutoTokenizer
import numpy as np



In [24]:
# ---------------- Args ----------------
dataset_name = "yield-v1-factualnovelty-rl-min3"




In [25]:
# ---------------- Config ----------------
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]
data_path = os.path.join(proj_store, "data")
models_folderpath = config["paths"]["models"]

DATASET_DIR = os.path.join(data_path, dataset_name)
OUTPUT_DIR = os.path.join(proj_store, "output", "dataset-analysis")
os.makedirs(OUTPUT_DIR, exist_ok=True)






In [26]:
# ---------------- Main ----------------

domain_counts = defaultdict(int)
split_counts = {"train": 0, "dev": 0, "test": 0}
domain_token_counts = defaultdict(list)
domain_novelty = defaultdict(list)
domain_return = defaultdict(list)

# Walk through splits
for split in ["train", "dev", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_dir):
        continue
    
    for filename in os.listdir(split_dir):
        if not filename.endswith(".jsonl"):
            continue
        
        file_path = os.path.join(split_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                    domain = data.get("domain", "unknown")
                    messages = data.get("messages", [])
                    domain_counts[domain] += 1
                    split_counts[split] += 1

                    # Rewards
                    factual_novelty = data.get("factual_novelty_score")
                    return_to_go = data.get("return_to_go")

                    if factual_novelty is not None:
                        domain_novelty[domain].append(factual_novelty)
                    if return_to_go is not None:
                        domain_return[domain].append(return_to_go)

                    # Apply chat template and tokenize
                    chat_text = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=False
                    )
                    tokenized = tokenizer(chat_text, add_special_tokens=False)
                    token_count = len(tokenized["input_ids"])

                    domain_token_counts[domain].append(token_count)

                except Exception as e:
                    print(f"Warning: Could not process a line in {file_path}: {e}")


# ---- 1. Domain totals ----
domain_rows = []
for domain, count in domain_counts.items():
    tokens = domain_token_counts.get(domain, [])
    novelty = domain_novelty.get(domain, [])
    returns = domain_return.get(domain, [])

    total_tokens = int(np.sum(tokens)) if tokens else 0
    avg_tokens = float(np.mean(tokens)) if tokens else 0
    max_tokens = int(np.max(tokens)) if tokens else 0
    min_tokens = int(np.min(tokens)) if tokens else 0

    avg_novelty = float(np.mean(novelty)) if novelty else 0
    min_novelty = float(np.min(novelty)) if novelty else 0
    max_novelty = float(np.max(novelty)) if novelty else 0

    avg_return = float(np.mean(returns)) if returns else 0
    min_return = float(np.min(returns)) if returns else 0
    max_return = float(np.max(returns)) if returns else 0

    domain_rows.append({
        "domain": domain,
        "count": count,
        "total_tokens": total_tokens,
        "avg_tokens": round(avg_tokens, 2),
        "max_tokens": max_tokens,
        "min_tokens": min_tokens,
        "avg_factual_novelty": round(avg_novelty, 3),
        "min_factual_novelty": round(min_novelty, 3),
        "max_factual_novelty": round(max_novelty, 3),
        "avg_return_to_go": round(avg_return, 3),
        "min_return_to_go": round(min_return, 3),
        "max_return_to_go": round(max_return, 3)
    })

df_domains = pd.DataFrame(domain_rows).sort_values(by="count", ascending=False)

# Add total row
total_row = {
    "domain": "TOTAL",
    "count": df_domains["count"].sum(),
    "total_tokens": df_domains["total_tokens"].sum(),
    "avg_tokens": round(df_domains["avg_tokens"].mean(), 2),
    "max_tokens": df_domains["max_tokens"].max(),
    "min_tokens": df_domains["min_tokens"].min(),
    "avg_factual_novelty": round(df_domains["avg_factual_novelty"].mean(), 3),
    "min_factual_novelty": round(df_domains["min_factual_novelty"].min(), 3),
    "max_factual_novelty": round(df_domains["max_factual_novelty"].max(), 3),
    "avg_return_to_go": round(df_domains["avg_return_to_go"].mean(), 3),
    "min_return_to_go": round(df_domains["min_return_to_go"].min(), 3),
    "max_return_to_go": round(df_domains["max_return_to_go"].max(), 3)
}
df_domains.loc[len(df_domains)] = total_row






In [27]:
# ---- 2. Split totals ----
df_splits = pd.DataFrame(list(split_counts.items()), columns=["split", "count"])
df_splits.loc[len(df_splits)] = ["TOTAL", df_splits["count"].sum()]


In [28]:
# ---- 3. Export both ----

df_domains.to_csv(os.path.join(OUTPUT_DIR, f"{dataset_name}-domain-counts.csv"))
df_splits.to_csv(os.path.join(OUTPUT_DIR, f"{dataset_name}-split-counts.csv"), index=False)

# ---- 4. Print summary ----
print(f"Saved results to: {OUTPUT_DIR}")

display("Domain counts:", df_domains.head())
display("Split totals:", df_splits)



Saved results to: /data/sequential-ieas/output/dataset-analysis


'Domain counts:'

,domain,count,total_tokens,avg_tokens,max_tokens,min_tokens,avg_factual_novelty,min_factual_novelty,max_factual_novelty,avg_return_to_go,min_return_to_go,max_return_to_go
2,judicial_proceedings,48679,14920246,306.50,512,91,0.487,0.0,18.0,4.686,0.0,40.440
1,oral_history,42408,12156466,286.66,512,83,1.627,0.0,223.0,13.587,0.0,452.988
0,academic_interviews,11036,2255573,204.38,512,81,0.296,0.0,29.0,2.543,0.0,36.515
3,journalistic_investigations,1331,414870,311.70,511,99,1.314,0.0,26.0,8.689,0.0,117.114
4,TOTAL,103454,29747155,277.31,512,81,0.931,0.0,223.0,7.376,0.0,452.988


'Split totals:'

,split,count
0,train,83181
1,dev,9988
2,test,10285
3,TOTAL,103454
